# Week 5: Customer Sales Analysis
**Tools:** Pandas, NumPy, Matplotlib

This notebook performs loading, cleaning, string and datetime transformations, conditional filtering, aggregations, merging, pivot tables, churn analysis, visualizations and business insights.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('.')
sales = pd.read_csv(BASE / 'sales_data.csv')
customers = pd.read_csv(BASE / 'customer_data.csv')
churn = pd.read_csv(BASE / 'customer_churn.csv')
print('Sales:', sales.shape, '| Customer profile:', customers.shape, '| Churn:', churn.shape)
display(sales.head())
display(churn.head())

## 1. Data quality and cleaning

In [ ]:
sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
for col in ['Quantity', 'Price', 'Total_Sales']:
    sales[col] = pd.to_numeric(sales[col], errors='coerce')
sales['Product'] = sales['Product'].astype('string').str.strip().str.title()
sales['Region'] = sales['Region'].astype('string').str.strip().str.title()
sales['Customer_ID'] = sales['Customer_ID'].astype('string').str.strip().str.upper()
print('Missing values before cleaning:\\n', sales.isna().sum())
sales = sales.dropna(subset=['Date','Product','Quantity','Price','Customer_ID','Region'])
sales = sales.drop_duplicates()
sales = sales[(sales['Quantity'] >= 0) & (sales['Price'] >= 0)].copy()
sales['Total_Sales'] = sales['Quantity'] * sales['Price']
print('Clean sales shape:', sales.shape)

## 2. Filtering and datetime features

In [ ]:
# AND condition: North region AND revenue above the dataset median
and_filter = sales[(sales['Region'] == 'North') & (sales['Total_Sales'] > sales['Total_Sales'].median())]
# OR condition: Phone OR Laptop
or_filter = sales[sales['Product'].str.contains('Phone|Laptop', case=False, na=False)]
sales['Year'] = sales['Date'].dt.year
sales['Month'] = sales['Date'].dt.month
sales['Day'] = sales['Date'].dt.day
sales['Year_Month'] = sales['Date'].dt.to_period('M').astype(str)
print('AND filtered rows:', len(and_filter), '| OR filtered rows:', len(or_filter))
display(sales[['Date','Year','Month','Day','Year_Month']].head())

## 3. Customer profile and merging

In [ ]:
# Build a customer profile lookup from the sales data
customer_profile = (sales.groupby('Customer_ID')
    .agg(Region=('Region', lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0]),
         First_Purchase=('Date','min'), Last_Purchase=('Date','max'),
         Order_Count=('Date','count'), Lifetime_Sales=('Total_Sales','sum'))
    .reset_index())
# Many sales rows to one customer profile row
sales_with_customer = sales.merge(customer_profile, on='Customer_ID', how='left',
                                  suffixes=('', '_Profile'), validate='many_to_one')
display(sales_with_customer.head())
print('Merged rows:', len(sales_with_customer))

## 4. Aggregations and pivot table

In [ ]:
monthly_summary = sales.groupby('Year_Month').agg(
    Revenue=('Total_Sales','sum'), Units=('Quantity','sum'), Transactions=('Customer_ID','count'))
product_summary = sales.groupby('Product').agg(
    Revenue=('Total_Sales','sum'), Units=('Quantity','sum'), Transactions=('Customer_ID','count')).sort_values('Revenue', ascending=False)
region_summary = sales.groupby('Region').agg(
    Revenue=('Total_Sales','sum'), Customers=('Customer_ID','nunique')).sort_values('Revenue', ascending=False)
customer_summary = sales.groupby('Customer_ID').agg(
    Lifetime_Value=('Total_Sales','sum'), Purchases=('Date','count'), Units=('Quantity','sum')).sort_values('Lifetime_Value', ascending=False)
region_product_pivot = pd.pivot_table(sales, index='Region', columns='Product',
    values='Total_Sales', aggfunc='sum', fill_value=0, margins=True)
display(monthly_summary, product_summary, region_summary, customer_summary.head(10), region_product_pivot)

## 5. Customer churn / retention-risk analysis

In [ ]:
churn['Churn'] = pd.to_numeric(churn['Churn'], errors='coerce')
churn['Tenure'] = pd.to_numeric(churn['Tenure'], errors='coerce')
churn = churn.dropna(subset=['Churn','Tenure']).drop_duplicates()
churn_rate = churn['Churn'].mean()
contract_churn = churn.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
print(f'Churn rate: {churn_rate:.1%}')
display(contract_churn)
# IDs across the two supplied files use different formats; do not join them without a verified key.
print('Sales IDs sample:', sales['Customer_ID'].head().tolist())
print('Churn IDs sample:', churn['CustomerID'].head().tolist())

## 6. Visualizations

In [ ]:
product_summary['Revenue'].plot(kind='bar', figsize=(9,5), title='Revenue by Product')
plt.xlabel('Product'); plt.ylabel('Revenue (INR)'); plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

monthly_summary['Revenue'].plot(kind='line', marker='o', figsize=(9,5), title='Monthly Revenue Trend')
plt.xlabel('Month'); plt.ylabel('Revenue (INR)'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

region_summary['Revenue'].plot(kind='pie', autopct='%1.1f%%', figsize=(7,7), title='Regional Revenue Share')
plt.ylabel(''); plt.tight_layout(); plt.show()

contract_churn.mul(100).plot(kind='bar', figsize=(8,5), title='Churn Rate by Contract Type')
plt.xlabel('Contract'); plt.ylabel('Churn Rate (%)'); plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

## 7. Key insights and business recommendations

In [ ]:
total_revenue = sales['Total_Sales'].sum()
top_customer = customer_summary.index[0]
top_product = product_summary.index[0]
top_region = region_summary.index[0]
best_month = monthly_summary['Revenue'].idxmax()
print(f'Total revenue: Rs. {total_revenue:,.2f}')
print(f'Unique customers: {sales.Customer_ID.nunique()}')
print(f'Top customer: {top_customer} | LTV: Rs. {customer_summary.iloc[0].Lifetime_Value:,.2f}')
print(f'Top product: {top_product}')
print(f'Top region: {top_region}')
print(f'Best month: {best_month}')
print(f'Churn dataset churn rate: {churn_rate:.1%}')
print('Recommendation: reward high-value customers; tailor regional promotions; investigate contract groups with higher churn.')